
# Extend the EDA Helper with Visual Diagnostics

In this lesson you add plotting to your EDA helper. In the previous lesson the LLM generated pandas code directly. We could do the same for plots, but raw plotting code generated by an LLM is often inconsistent and unreliable.

In this lesson we solve that problem using **tool use**.



> Tool Use - In Lesson 1, the LLM wrote *all* the code. Here we split responsibilities: you write the plotting logic , the LLM writes only the *orchestration* code that calls your functions. This is called tool use and is a foundational pattern in LLM-powered applications.

## 1 — Setup

Same setup as Lesson 1, with two additions: `matplotlib` and `seaborn` are now imported because our plotting functions will use them directly.

In [1]:
%pip install -q google-genai pandas matplotlib seaborn scikit-learn python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import re
from google import genai
from google.genai import types
from dotenv import load_dotenv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings

warnings.filterwarnings("ignore", category=FutureWarning)


load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
df = pd.read_csv("../data/hr_analytics.csv")
print(df.shape)

## 2 — The Problem with Raw LLM Plotting

Let's ask the LLM to generate plots the same way we generated pandas code i.e. with a simple prompt.

We'll run the **same question twice** and compare what comes back. This demonstrates the core problem: raw LLM plotting code is non-deterministic. The same prompt can produce a seaborn chart on one run and a matplotlib chart on the next, with different styling or  different axis labels.

This gives the LLM maximum freedom but leads to inconsistent, unpredictable output. We'll fix this in Section 3

In [ ]:
NAIVE_PLOT_PROMPT = (
    "Write Python code to answer the question using the existing DataFrame variable `df`. "
    "If the question involves visualization, use matplotlib or seaborn to visualize it. "
    "Display any plots with plt.show(). "
    "If the answer is tabular, store it in `result_df`. "
    "Do NOT create a new DataFrame from scratch. "
    "Do NOT include import statements. "
    "Return ONLY executable Python code, no explanations."
)


def eda_helper_naive(question, frame: pd.DataFrame, show_code=False):
    """Naive version: LLM writes raw plotting code."""
    prompt = (
        f"Columns: {list(frame.columns)}\n"
        f"Dtypes:\n{frame.dtypes.to_string()}\n\n"
        f"Sample (10 rows):\n{frame.head(10).to_string()}\n\n"
        f"Question: {question}"
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={"system_instruction": NAIVE_PLOT_PROMPT},
    )

    text = (response.text or "").strip()
    match = re.search(
        r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE
    )
    code = match.group(1).strip() if match else text

    if show_code:
        print("#Generated Code#")
        print(code)
        print("---\n")

    env = {"pd": pd, "df": frame.copy(), "plt": plt, "sns": sns}
    exec(code, env, env)
    return env.get("result_df")

In [ ]:
eda_helper_naive(
    "Plot the distribution of age",
    df,
    show_code=True,
)

## 3 — Build Your Own Plotting Functions

Let's define the plotting functions ourselves.


In [ ]:
# For understanding the shape of a numeric column: normally distributed? Skewed? Does it have outliers?
def plot_distribution(df, column, title=None):
    """Plot histogram with KDE for a numeric column."""
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(df[column], kde=True, ax=ax, edgecolor="black")
    ax.set_title(title or f"Distribution of {column}")
    plt.tight_layout()
    plt.show()


# How many rows fall into each category.
def plot_bar_counts(df, column, title=None):
    """Plot value counts as a horizontal bar chart."""
    fig, ax = plt.subplots(figsize=(8, 4))
    df[column].value_counts().plot(kind="barh", ax=ax, edgecolor="black")
    ax.set_title(title or f"Value Counts: {column}")
    plt.tight_layout()
    plt.show()


# Which pairs of numeric columns move together.
def plot_correlation(df, title="Correlation Matrix"):
    """Plot correlation heatmap for all numeric columns."""
    numeric_df = df.select_dtypes(include="number")
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


# Compare the spread of a numeric column across different groups.
def plot_boxplot(df, column, by=None, title=None):
    """Plot boxplot, optionally grouped by a categorical column."""
    fig, ax = plt.subplots(figsize=(8, 4))
    if by:
        sns.boxplot(data=df, x=by, y=column, ax=ax)
    else:
        sns.boxplot(data=df, y=column, ax=ax)
    ax.set_title(title or f"Boxplot: {column}" + (f" by {by}" if by else ""))
    plt.tight_layout()
    plt.show()


Quick sanity check — do these work on their own?

In [ ]:
plot_distribution(df, "age", title="Distribution of Age")

In [ ]:
plot_bar_counts(df, "department")

## 4 —  Register Tools as a Schema
To use structured tool calling, we describe each function formally. This is what we pass to the model.
The model reads this schema and returns a structured call: a function name and a typed argument dict.



In [ ]:
TOOLS_SCHEMA = types.Tool(
    functionDeclarations=[
        {
            "name": "plot_distribution",
            "description": "Plot histogram with KDE for a numeric column.",
            "parameters": {
                "type": "object",
                "properties": {"column": {"type": "string"}},
                "required": ["column"],
            },
        },
        {
            "name": "plot_bar_counts",
            "description": "Plot value counts as a bar chart for a categorical column.",
            "parameters": {
                "type": "object",
                "properties": {"column": {"type": "string"}},
                "required": ["column"],
            },
        },
        {
            "name": "plot_correlation",
            "description": "Plot correlation heatmap for all numeric columns.",
            "parameters": {
                "type": "object",
                "properties": {},
            },
        },
        {
            "name": "plot_boxplot",
            "description": "Plot boxplot of a numeric column, optionally grouped by a categorical column.",
            "parameters": {
                "type": "object",
                "properties": {
                    "column": {"type": "string"},
                    "by": {"type": "string"},
                },
                "required": ["column"],
            },
        },
    ]
)


## 5 - Build the Visual EDA Helper
The helper sends the question + column metadata to the model with the tool schema attached. The model returns a `function_call part` with a name and arguments, and we dispatch to the matching plotting function.


The dispatch dictionary is optional here, so we call the plotting helpers explicitly after reading the returned function name.


In [ ]:
def eda_visual_helper(question: str, frame: pd.DataFrame):
    """Ask an EDA question and dispatch the appropriate plotting tool."""
    prompt = (
        f"Columns: {list(frame.columns)}\n"
        f"Dtypes:\n{frame.dtypes.to_string()}\n\n"
        f"Sample (10 rows):\n{frame.head(10).to_string()}\n\n"
        f"Question: {question}"
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.0, seed=42, tools=[TOOLS_SCHEMA]
        ),
    )

    part = response.candidates[0].content.parts[0]
    tool_name = part.function_call.name
    args = dict(part.function_call.args)

    if tool_name == "plot_distribution":
        plot_distribution(df=frame.copy(), **args)
    elif tool_name == "plot_bar_counts":
        plot_bar_counts(df=frame.copy(), **args)
    elif tool_name == "plot_correlation":
        plot_correlation(df=frame.copy(), **args)
    elif tool_name == "plot_boxplot":
        plot_boxplot(df=frame.copy(), **args)
    else:
        raise ValueError(f"Unsupported tool requested: {tool_name}")
    return {"tool": tool_name, "args": args}

## 5 — Let's Try It Out

In [ ]:
# numeric column — should select plot_distribution
eda_visual_helper("Plot the distribution of age", df)

In [ ]:
# Same question again; code should be consistent now
eda_visual_helper("Plot the distribution of age", df)

- Run Categorical column

The LLM should recognise that `department` is a categorical column (it saw this in `df.dtypes`) and choose `plot_bar_counts` rather than `plot_distribution`.

In [ ]:
eda_visual_helper("Show the distribution of employees across departments", df)

- Correlation

The word *"correlation"* in the question should trigger `plot_correlation(df)`. 

In [ ]:
eda_visual_helper("Show the correlation between numeric columns", df)

In [ ]:
eda_visual_helper("Show monthly income grouped by department", df)

## 6 — Save Helpers for Reuse

The plotting functions and helper are saved to `eda_helpers.py` as they will be reused in future notebooks. This ensures that the code is organized and can be easily imported without needing to redefine anything.

In [ ]:
import inspect

# define this once in notebook so we can reuse the same JSON Schema in the export
TOOLS_SCHEMA_JSON = {
    "plot_distribution": {
        "type": "object",
        "properties": {"column": {"type": "string"}},
        "required": ["column"],
    },
    "plot_bar_counts": {
        "type": "object",
        "properties": {"column": {"type": "string"}},
        "required": ["column"],
    },
    "plot_correlation": {"type": "object", "properties": {}},
    "plot_boxplot": {
        "type": "object",
        "properties": {
            "column": {"type": "string"},
            "by": {"type": "string"},
        },
        "required": ["column"],
    },
}

components = [
    "import os",
    "import pandas as pd",
    "import matplotlib.pyplot as plt",
    "import seaborn as sns",
    "from dotenv import load_dotenv",
    "from google import genai",
    "from google.genai import types",
    "",
    "load_dotenv()",
    "client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))",
    "",
    f"TOOLS_SCHEMA_JSON = {repr(TOOLS_SCHEMA_JSON)}",
    "TOOLS_SCHEMA = types.Tool(",
    "    functionDeclarations=[",
    "        types.FunctionDeclaration(",
    "            name='plot_distribution',",
    "            description='Plot histogram with KDE for a numeric column.',",
    "            parametersJsonSchema=TOOLS_SCHEMA_JSON['plot_distribution'],",
    "        ),",
    "        types.FunctionDeclaration(",
    "            name='plot_bar_counts',",
    "            description='Plot value counts as a bar chart for a categorical column.',",
    "            parametersJsonSchema=TOOLS_SCHEMA_JSON['plot_bar_counts'],",
    "        ),",
    "        types.FunctionDeclaration(",
    "            name='plot_correlation',",
    "            description='Plot correlation heatmap for all numeric columns.',",
    "            parametersJsonSchema=TOOLS_SCHEMA_JSON['plot_correlation'],",
    "        ),",
    "        types.FunctionDeclaration(",
    "            name='plot_boxplot',",
    "            description='Plot boxplot of a numeric column, optionally grouped by a categorical column.',",
    "            parametersJsonSchema=TOOLS_SCHEMA_JSON['plot_boxplot'],",
    "        ),",
    "    ]",
    ")",
    "",
    inspect.getsource(plot_distribution),
    inspect.getsource(plot_bar_counts),
    inspect.getsource(plot_correlation),
    inspect.getsource(plot_boxplot),
    "",
    inspect.getsource(eda_visual_helper),
]

with open("eda_visual_helpers.py", "w", encoding="utf-8") as f:
    f.write("\n".join(components))

print("Saved eda_visual_helpers.py")